# Sparse selection and stability

This notebook runs the Phase 3 sparse-selection and stability stage. During automated validation it resolves placeholder paths to the deterministic demo dataset; once local real-data paths are configured it runs against the real case-study artifacts.

## Notebook Overview

### Purpose
Sparse feature selection via de-biased LASSO. Select stable features with high selection frequency across bootstrap samples. Filter via HC3 heteroscedasticity criterion.

### Inputs
- Feature DataFrame from stage 05 (~400-1100 features)
- `df_pheno`: Phenotype/outcome variable
- De-biased LASSO implementation: `src/rfm_pipeline/LASSO_to_OLS_v9.ipynb`
- HC3 robust variance filter: from configuration
- Stability parameters: repetitions, selection frequency threshold

### Outputs
- `selected_features`: Final stable feature set after LASSO + stability filtering (~50-100 features)
- Stability summary: `outputs/06_sparse/stability_summary.csv` (selection frequency, HC3 scores)
- HC3 filter report: features passing variance criterion
- Stage 6 handoff CSV: `outputs/06_sparse/features_06_stable_selected.csv`
- LASSO path and stability curves (optional plot outputs)

### Parameters & Configuration
- **LASSO repetitions**: Number of bootstrap samples for stability analysis. Default 100. Higher = more stable estimates but slower.
- **Selection frequency threshold**: Minimum proportion of runs a feature must be selected. Default 0.5 (selected in 50+ out of 100 runs).
- **HC3 percentile**: Heteroscedasticity criterion threshold (robust variance filter). Default 75th percentile. Features with HC3 score > threshold are kept.
- **Regularization strength**: LASSO λ parameter. Default: cross-validated 1SE rule. Can be set manually.

### Expected Output
Notebook completes in ~60-180 seconds (depends on repetitions and feature count).
- Final selected features: typically 50-100 stable features
- Stability summary shows selection frequency distribution (should be right-skewed, many features with freq >> threshold)
- HC3 scores printed; most selected features have HC3 < threshold (lower variance)
- Stage 6 handoff CSV written with stable feature set

---


In [ ]:
import os
import sys
from pathlib import Path


def _find_study_root() -> Path:
    env_root = os.environ.get("RFM_STUDY_ROOT")
    if env_root:
        candidate = Path(env_root).resolve()
        if (candidate / "configs").exists():
            return candidate
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "configs").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError(
        "Could not locate the study root. Set RFM_STUDY_ROOT or run from the study directory."
    )


REPO_ROOT = _find_study_root()
SRC_DIR = REPO_ROOT / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:
from rfm_pipeline import build_manuscript_notebook_context, run_sparse_selection_stability_stage

context = build_manuscript_notebook_context(REPO_ROOT, "06_sparse_selection_and_stability.ipynb")
result = run_sparse_selection_stability_stage(context)
result.sparse_selection.summary

### Step 1: Run De-Biased LASSO with Stability Analysis

Select stable features via repeated LASSO on bootstrap samples. Use de-biased LASSO to correct for selection bias.

**Process:**
1. For each of B bootstrap samples (~100):
   a. Fit LASSO on bootstrap resample
   b. Record which features are selected
   c. Estimate coefficients using de-biased correction
2. Compute feature selection frequency (e.g., selected in 50+ of 100 runs)
3. Keep only stable features (freq >= threshold)

**Parameters:**
- `lasso_repetitions`: Number of bootstrap samples (default 100)
- `selection_frequency_threshold`: Min frequency to keep (default 0.5, i.e., 50%)
- `lambda_rule`: LASSO regularization strength. 'cv1se' (default), 'cvmin', or manual value

**Output:**
- `stability_summary`: Selection frequency for each feature
- `selected_features_stable`: Final stable feature set (~50-100 features)

In [ ]:
result.sparse_selection.final_stable_support.head()

### Step 2: Apply HC3 Heteroscedasticity Filter

Filter stable features via robust variance criterion to remove inflated-variance features.

**HC3 filter:**
1. Fit model on stable features
2. Compute HC3 robust variance estimates for each coefficient
3. Keep features with HC3 score <= threshold (lower variance = better)
4. Typically removes 10-20% of stable features

**Output:**
- Final selected features: stable + HC3-filtered (~50-100 features)
- Handoff CSV ready for OLS